# **$\mathcal{R}_0$ in systems with a nonlinear disease-free equilibrium**

In within-host models with an immune response, the disease-free equilibrium (DFE) can be a coupled nonlinear system: for example, helper and regulatory T lymphocytes that regulate each other, and effector cells activated by both. Solving it at once with `sympy.solve` may never finish.

`pyR0compute` solves it **block by block**. With the infected compartments set to zero, the equations of the uninfected compartments are split into the strongly connected components of their dependency graph and solved in topological order, for example $[E] \to [H, R] \to [C] \to [B]$. Values already solved enter the following blocks as symbols $X^*$ (and not as their explicit expressions), which keeps the expressions small. Short values, such as $E^* = g_E/d_E$, are written out in full; the others stay as $X^*$ in `R0_compact` and are defined in `dfe_definitions`.

This notebook contains:

1. The T-helper lymphocyte model of Cuesta-Herrera et al. (2025), *Math. Biosci. Eng.* 22(11):2807-2825, [doi:10.3934/mbe.2025103](https://doi.org/10.3934/mbe.2025103). Eq. (2.4) and the $\mathcal{R}_0^*$ values of Figure 3 are reproduced.
2. A 7-equation within-host model with an immune response, whose DFE is nonlinear.
3. A DFE without closed form (root of a quintic), kept implicit and computed numerically.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yvan-BM-Git/pyR0compute/blob/main/examples/03_nonlinear_systems.ipynb)

In [1]:
%pip install pyR0compute                      # Install pyR0compute from PyPI

In [2]:
import time                                   # Measure computation times
import numpy as np                            # Numerical computation
import sympy as sp                            # Symbolic computation
from pyr0compute import R0Model               # Main class of the library
from IPython.display import Markdown, display # Show equations in LaTeX

## 1. T-helper lymphocyte model (Cuesta-Herrera et al., 2025)

System (2.2) of the paper: susceptible epithelial cells $E$, infected cells $I$, virus particles $V$ and activated helper T lymphocytes $T_h$.

$$
\begin{cases}
\dfrac{dE}{dt} = \lambda - dE - \kappa EV \\[6pt]
\dfrac{dI}{dt} = \kappa EV - \alpha I - \beta T_h I \\[6pt]
\dfrac{dV}{dt} = \nu I - \mu V - \tau T_h V \\[6pt]
\dfrac{dT_h}{dt} = b - cT_h + \gamma I T_h
\end{cases}
\qquad
\mathcal{R}_0^* = \frac{\kappa\lambda\nu/d}{\left(\alpha + \beta\,\frac{b}{c}\right)\left(\mu + \tau\,\frac{b}{c}\right)}
\quad \text{(Eq. 2.4)}
$$

Here the DFE is linear ($E_0 = \lambda/d$, $T_{h0} = b/c$), so everything is written explicitly.

In [3]:
model_th = R0Model('''
    dE/dt  = lambda - d*E - kappa*E*V
    dI/dt  = kappa*E*V - alpha*I - beta*Th*I
    dV/dt  = nu*I - mu*V - tau*Th*V
    dTh/dt = b - c*Th + gamma*I*Th
''', infected=["I", "V"])

print("DFE:", model_th.dfe)
print("R0* =", model_th.R0)

# Comparison with Eq. (2.4) of the paper
p = {s.name: s for s in model_th.parameters}
eq_24 = (p["kappa"] * p["lambda"] * p["nu"] / p["d"]) / (
    (p["alpha"] + p["beta"] * p["b"] / p["c"]) * (p["mu"] + p["tau"] * p["b"] / p["c"]))
print("Matches Eq. (2.4):", sp.simplify(model_th.R0 - eq_24) == 0)

DFE: {E: lambda/d, I: 0, V: 0, Th: b/c}
R0* = c**2*kappa*lambda*nu/(d*(alpha*c + b*beta)*(b*tau + c*mu))
Matches Eq. (2.4): True


### Values of Figure 3

With the values of Table 1 (system 2.2) the paper reports $\mathcal{R}_0^* = 6.0303\times10^3$ (Figure 3a). In Figure 3c, $\beta$ and $\tau$ increase and $d$, $\lambda$, $\kappa$ and $b$ change; there $\mathcal{R}_0^* = 19.8920$.

In [4]:
table1 = {"lambda": 5e5, "d": 0.1, "kappa": 1e-7, "alpha": 0.1, "beta": 1e-7, "nu": 995,
          "mu": 0.5, "tau": 1e-7, "b": 5e4, "c": 0.1, "gamma": 1e-7}
fig3c = dict(table1, **{"lambda": 1e6, "beta": 1.1e-7, "tau": 1e-4, "d": 0.2, "kappa": 1e-8, "b": 2e4})

print(f"Figure 3a: R0* = {model_th.R0_numeric(table1):.4f}   (paper: 6.0303e3)")
print(f"Figure 3c: R0* = {model_th.R0_numeric(fig3c):.4f}   (paper: 19.8920)")

Figure 3a: R0* = 6030.3030   (paper: 6.0303e3)
Figure 3c: R0* = 19.8920   (paper: 19.8920)


### Condition (A5) of van den Driessche and Watmough

$\mathcal{R}_0$ is a threshold only if the DFE is stable in the absence of infection: the Jacobian of the uninfected equations with respect to the uninfected compartments, evaluated at the DFE, must have eigenvalues with negative real part. In the proof of Theorem 1 of the paper, $-d$ and $-c$ appear. `dfe_stability()` returns these symbolic eigenvalues when the block is triangular, and checks stability on random parameter samples.

In [5]:
stab = model_th.dfe_stability(n_samples=50)
print("Eigenvalues of the uninfected block:", stab["eigenvalues"])
print(f"Stable in {stab['stable']} of {stab['feasible']} feasible samples")

Eigenvalues of the uninfected block: [-d, -c]
Stable in 50 of 50 feasible samples


## 2. Seven-equation model with an immune response

Target cells $E$, infected cells $I$, virus $V$, helper T lymphocytes $H$, cytotoxic T lymphocytes $C$, B cells $B$ and regulatory T lymphocytes $R$. The interactions are saturating (Michaelis-Menten type):

$$
\begin{cases}
E' = g_E - \left(d_E + \tau_E \dfrac{V}{V+\zeta_{VE}}\right)E \\[8pt]
I' = \tau_E \dfrac{VE}{V+\zeta_{VE}} - \left(d_I + \tau_I \dfrac{C}{C+\zeta_{CI}}\right)I \\[8pt]
V' = \nu I - \left(d_V + \tau_V \dfrac{B}{B+\zeta_{BV}}\right)V \\[8pt]
H' = g_H - \left(d_H - \tau_H \dfrac{I}{I+\zeta_{IH}} + \rho_H \dfrac{R}{R+\zeta_{RH}}\right)H \\[8pt]
C' = g_C - \left(d_C - \tau_C \dfrac{H}{H+\zeta_{HC}} + \rho_C \dfrac{R}{R+\zeta_{RC}}\right)C \\[8pt]
B' = g_B - \left(d_B - \tau_B \dfrac{H}{H+\zeta_{HB}} + \rho_B \dfrac{R}{R+\zeta_{RB}}\right)B \\[8pt]
R' = g_R - \left(d_R - \tau_R \dfrac{H}{H+\zeta_{HR}} + \rho_R\right)R
\end{cases}
$$

With $I = V = 0$, the DFE couples $H$ and $R$ (a quadratic with a single positive root), and $C$, $B$ depend on both. In the model written as text, $\zeta$ is written `z`.

In [6]:
t0 = time.time()
model = R0Model('''
    dE/dt = g_E - (d_E + tau_E*V/(V + z_VE))*E
    dI/dt = tau_E*V*E/(V + z_VE) - (d_I + tau_I*C/(C + z_CI))*I
    dV/dt = nu*I - (d_V + tau_V*B/(B + z_BV))*V
    dH/dt = g_H - (d_H - tau_H*I/(I + z_IH) + rho_H*R/(R + z_RH))*H
    dC/dt = g_C - (d_C - tau_C*H/(H + z_HC) + rho_C*R/(R + z_RC))*C
    dB/dt = g_B - (d_B - tau_B*H/(H + z_HB) + rho_B*R/(R + z_RB))*B
    dR/dt = g_R - (d_R - tau_R*H/(H + z_HR) + rho_R)*R
''', infected=["I", "V"])
print(f"Model built in {time.time() - t0:.1f} s ({len(model.parameters)} parameters)")

Model built in 7.2 s (34 parameters)


### Compact DFE and solving order

`dfe_compact` shows the DFE with the symbols $X^*$ (in text they are called `X_star`, so that `S_star*beta` is not read as a power). `dfe_definitions` defines them in the order in which they are solved: first the coupled block $\{H, R\}$, then $C$ and $B$ as functions of $H^*$ and $R^*$.

In [7]:
print("Compact DFE:")
for x, value in model.dfe_compact.items():
    print(f"  {x} = {value}")

print("\nDefinitions, in solving order:")
for s, value in model.dfe_definitions.items():
    text = str(value)
    print(f"  {s} = {text[:110] + ' ...' if len(text) > 110 else text}")

Compact DFE:
  E = g_E/d_E
  I = 0
  V = 0
  H = H_star
  C = C_star
  B = B_star
  R = R_star

Definitions, in solving order:
  H_star = (-d_H*g_R*z_HR + d_R*g_H*z_RH + g_H*rho_R*z_RH - g_H*tau_R*z_RH - g_R*rho_H*z_HR + (-(d_H*d_R*z_HR*z_RH - d_H* ...
  R_star = -(d_H*d_R*z_HR*z_RH - d_H*g_R*z_HR + d_H*rho_R*z_HR*z_RH + d_R*g_H*z_RH - g_H*g_R + g_H*rho_R*z_RH - g_H*tau_R ...
  C_star = g_C*(H_star + z_HC)*(R_star + z_RC)/(H_star*R_star*d_C + H_star*R_star*rho_C - H_star*R_star*tau_C + H_star*d_ ...
  B_star = g_B*(H_star + z_HB)*(R_star + z_RB)/(H_star*R_star*d_B + H_star*R_star*rho_B - H_star*R_star*tau_B + H_star*d_ ...


### Compact $\mathcal{R}_0$

The Jacobians $F$ and $V$ are evaluated at the compact DFE, so $\mathcal{R}_0$ is simplified while it is still small:

$$
\mathcal{R}_0 = \frac{g_E\,\nu\,\tau_E}{d_E\,\zeta_{VE}\,k_I\,k_V},
\qquad
k_I = d_I + \tau_I\frac{C^*}{C^*+\zeta_{CI}},
\qquad
k_V = d_V + \tau_V\frac{B^*}{B^*+\zeta_{BV}}.
$$

$k_I$ is the removal rate of infected cells, increased by baseline cytotoxic T cells, and $k_V$ that of the virus, increased by neutralization by B cells. `model.R0` returns the same quantity with the explicit values substituted, but it is an expression with tens of thousands of operations.

In [8]:
display(Markdown(f"$$\\mathcal{{R}}_0 = {model.latex()}$$"))
print(f"Explicit R0 (DFE substituted): {sp.count_ops(model.R0)} operations")

$$\mathcal{R}_0 = \frac{g_{E} \nu \tau_{E} \left(B^{*} + z_{BV}\right) \left(C^{*} + z_{CI}\right)}{d_{E} z_{VE} \left(B^{*} d_{V} + B^{*} \tau_{V} + d_{V} z_{BV}\right) \left(C^{*} d_{I} + C^{*} \tau_{I} + d_{I} z_{CI}\right)}$$

Explicit R0 (DFE substituted): 23948 operations


The step-by-step LaTeX report also uses the compact form. With `style="document"` and `standalone=True` you get a compilable file for a paper.

In [9]:
display(Markdown(model.report_latex("markdown")))

## pyR0compute: next-generation matrix method

**Variables:** $E, I, V, H, C, B, R$  
**Infected compartments:** $I, V$  
**Parameters (detected automatically):** $d_{B}, d_{C}, d_{E}, d_{H}, d_{I}, d_{R}, d_{V}, g_{B}, g_{C}, g_{E}, g_{H}, g_{R}, \nu, \rho_{B}, \rho_{C}, \rho_{H}, \rho_{R}, \tau_{B}, \tau_{C}, \tau_{E}, \tau_{H}, \tau_{I}, \tau_{R}, \tau_{V}, z_{BV}, z_{CI}, z_{HB}, z_{HC}, z_{HR}, z_{IH}, z_{RB}, z_{RC}, z_{RH}, z_{VE}$

### Model

$$
\begin{aligned}
\frac{dE}{dt} &= - E \left(\frac{V \tau_{E}}{V + z_{VE}} + d_{E}\right) + g_{E} \\
\frac{dI}{dt} &= \frac{E V \tau_{E}}{V + z_{VE}} - I \left(\frac{C \tau_{I}}{C + z_{CI}} + d_{I}\right) \\
\frac{dV}{dt} &= I \nu - V \left(\frac{B \tau_{V}}{B + z_{BV}} + d_{V}\right) \\
\frac{dH}{dt} &= - H \left(- \frac{I \tau_{H}}{I + z_{IH}} + \frac{R \rho_{H}}{R + z_{RH}} + d_{H}\right) + g_{H} \\
\frac{dC}{dt} &= - C \left(- \frac{H \tau_{C}}{H + z_{HC}} + \frac{R \rho_{C}}{R + z_{RC}} + d_{C}\right) + g_{C} \\
\frac{dB}{dt} &= - B \left(- \frac{H \tau_{B}}{H + z_{HB}} + \frac{R \rho_{B}}{R + z_{RB}} + d_{B}\right) + g_{B} \\
\frac{dR}{dt} &= - R \left(- \frac{H \tau_{R}}{H + z_{HR}} + d_{R} + \rho_{R}\right) + g_{R}
\end{aligned}
$$

### New-infection terms (automatic)

$$
\begin{aligned}
\mathcal{F}_{I} &= \frac{E V \tau_{E}}{V + z_{VE}} \\
\mathcal{F}_{V} &= 0
\end{aligned}
$$

### Term classification

| Equation | Term | Class | Reason |
|---|---|:---:|---|
| $dI/dt$ | $- I d_{I}$ | $\mathcal{V}$ | negative term |
| $dI/dt$ | $\frac{E V \tau_{E}}{V + z_{VE}}$ | $\mathcal{F}$ | transfer from uninfected E |
| $dI/dt$ | $- \frac{C I \tau_{I}}{C + z_{CI}}$ | $\mathcal{V}$ | negative term |
| $dV/dt$ | $I \nu$ | $\mathcal{V}$ | transfer from infected I |
| $dV/dt$ | $- V d_{V}$ | $\mathcal{V}$ | negative term |
| $dV/dt$ | $- \frac{B V \tau_{V}}{B + z_{BV}}$ | $\mathcal{V}$ | negative term |

### Transition terms

$$
\begin{aligned}
\mathcal{V}_{I} &= \frac{C I \tau_{I}}{C + z_{CI}} + I d_{I} \\
\mathcal{V}_{V} &= \frac{B V \tau_{V}}{B + z_{BV}} - I \nu + V d_{V}
\end{aligned}
$$

### Disease-free equilibrium

$$
\begin{aligned}
E &= \frac{g_{E}}{d_{E}} \\
I &= 0 \\
V &= 0 \\
H &= H^{*} \\
C &= C^{*} \\
B &= B^{*} \\
R &= R^{*}
\end{aligned}
$$

The values marked with $^*$ are found block by block, in this order:

$$
\begin{aligned}
H^{*} &= \frac{- d_{H} g_{R} z_{HR} + d_{R} g_{H} z_{RH} + g_{H} \rho_{R} z_{RH} - g_{H} \tau_{R} z_{RH} - g_{R} \rho_{H} z_{HR} + \left(- \frac{d_{H} d_{R} z_{HR} z_{RH} - d_{H} g_{R} z_{HR} + d_{H} \rho_{R} z_{HR} z_{RH} + d_{R} g_{H} z_{RH} - g_{H} g_{R} + g_{H} \rho_{R} z_{RH} - g_{H} \tau_{R} z_{RH} - g_{R} \rho_{H} z_{HR}}{2 \left(d_{H} d_{R} z_{HR} + d_{H} \rho_{R} z_{HR} + d_{R} g_{H} + d_{R} \rho_{H} z_{HR} + g_{H} \rho_{R} - g_{H} \tau_{R} + \rho_{H} \rho_{R} z_{HR}\right)} + \frac{\sqrt{d_{H}^{2} d_{R}^{2} z_{HR}^{2} z_{RH}^{2} + 2 d_{H}^{2} d_{R} g_{R} z_{HR}^{2} z_{RH} + 2 d_{H}^{2} d_{R} \rho_{R} z_{HR}^{2} z_{RH}^{2} + d_{H}^{2} g_{R}^{2} z_{HR}^{2} + 2 d_{H}^{2} g_{R} \rho_{R} z_{HR}^{2} z_{RH} + d_{H}^{2} \rho_{R}^{2} z_{HR}^{2} z_{RH}^{2} + 2 d_{H} d_{R}^{2} g_{H} z_{HR} z_{RH}^{2} + 4 d_{H} d_{R} g_{H} g_{R} z_{HR} z_{RH} + 4 d_{H} d_{R} g_{H} \rho_{R} z_{HR} z_{RH}^{2} - 2 d_{H} d_{R} g_{H} \tau_{R} z_{HR} z_{RH}^{2} + 2 d_{H} d_{R} g_{R} \rho_{H} z_{HR}^{2} z_{RH} + 2 d_{H} g_{H} g_{R}^{2} z_{HR} + 4 d_{H} g_{H} g_{R} \rho_{R} z_{HR} z_{RH} - 2 d_{H} g_{H} g_{R} \tau_{R} z_{HR} z_{RH} + 2 d_{H} g_{H} \rho_{R}^{2} z_{HR} z_{RH}^{2} - 2 d_{H} g_{H} \rho_{R} \tau_{R} z_{HR} z_{RH}^{2} + 2 d_{H} g_{R}^{2} \rho_{H} z_{HR}^{2} + 2 d_{H} g_{R} \rho_{H} \rho_{R} z_{HR}^{2} z_{RH} + d_{R}^{2} g_{H}^{2} z_{RH}^{2} + 2 d_{R} g_{H}^{2} g_{R} z_{RH} + 2 d_{R} g_{H}^{2} \rho_{R} z_{RH}^{2} - 2 d_{R} g_{H}^{2} \tau_{R} z_{RH}^{2} + 2 d_{R} g_{H} g_{R} \rho_{H} z_{HR} z_{RH} + g_{H}^{2} g_{R}^{2} + 2 g_{H}^{2} g_{R} \rho_{R} z_{RH} - 2 g_{H}^{2} g_{R} \tau_{R} z_{RH} + g_{H}^{2} \rho_{R}^{2} z_{RH}^{2} - 2 g_{H}^{2} \rho_{R} \tau_{R} z_{RH}^{2} + g_{H}^{2} \tau_{R}^{2} z_{RH}^{2} + 2 g_{H} g_{R}^{2} \rho_{H} z_{HR} + 2 g_{H} g_{R} \rho_{H} \rho_{R} z_{HR} z_{RH} + 2 g_{H} g_{R} \rho_{H} \tau_{R} z_{HR} z_{RH} + g_{R}^{2} \rho_{H}^{2} z_{HR}^{2}}}{2 \left(d_{H} d_{R} z_{HR} + d_{H} \rho_{R} z_{HR} + d_{R} g_{H} + d_{R} \rho_{H} z_{HR} + g_{H} \rho_{R} - g_{H} \tau_{R} + \rho_{H} \rho_{R} z_{HR}\right)}\right) \left(d_{H} d_{R} z_{HR} + d_{H} \rho_{R} z_{HR} + d_{R} g_{H} + d_{R} \rho_{H} z_{HR} + g_{H} \rho_{R} - g_{H} \tau_{R} + \rho_{H} \rho_{R} z_{HR}\right)}{d_{H} d_{R} z_{RH} + d_{H} g_{R} + d_{H} \rho_{R} z_{RH} - d_{H} \tau_{R} z_{RH} + g_{R} \rho_{H}} \\
R^{*} &= - \frac{d_{H} d_{R} z_{HR} z_{RH} - d_{H} g_{R} z_{HR} + d_{H} \rho_{R} z_{HR} z_{RH} + d_{R} g_{H} z_{RH} - g_{H} g_{R} + g_{H} \rho_{R} z_{RH} - g_{H} \tau_{R} z_{RH} - g_{R} \rho_{H} z_{HR}}{2 \left(d_{H} d_{R} z_{HR} + d_{H} \rho_{R} z_{HR} + d_{R} g_{H} + d_{R} \rho_{H} z_{HR} + g_{H} \rho_{R} - g_{H} \tau_{R} + \rho_{H} \rho_{R} z_{HR}\right)} + \frac{\sqrt{d_{H}^{2} d_{R}^{2} z_{HR}^{2} z_{RH}^{2} + 2 d_{H}^{2} d_{R} g_{R} z_{HR}^{2} z_{RH} + 2 d_{H}^{2} d_{R} \rho_{R} z_{HR}^{2} z_{RH}^{2} + d_{H}^{2} g_{R}^{2} z_{HR}^{2} + 2 d_{H}^{2} g_{R} \rho_{R} z_{HR}^{2} z_{RH} + d_{H}^{2} \rho_{R}^{2} z_{HR}^{2} z_{RH}^{2} + 2 d_{H} d_{R}^{2} g_{H} z_{HR} z_{RH}^{2} + 4 d_{H} d_{R} g_{H} g_{R} z_{HR} z_{RH} + 4 d_{H} d_{R} g_{H} \rho_{R} z_{HR} z_{RH}^{2} - 2 d_{H} d_{R} g_{H} \tau_{R} z_{HR} z_{RH}^{2} + 2 d_{H} d_{R} g_{R} \rho_{H} z_{HR}^{2} z_{RH} + 2 d_{H} g_{H} g_{R}^{2} z_{HR} + 4 d_{H} g_{H} g_{R} \rho_{R} z_{HR} z_{RH} - 2 d_{H} g_{H} g_{R} \tau_{R} z_{HR} z_{RH} + 2 d_{H} g_{H} \rho_{R}^{2} z_{HR} z_{RH}^{2} - 2 d_{H} g_{H} \rho_{R} \tau_{R} z_{HR} z_{RH}^{2} + 2 d_{H} g_{R}^{2} \rho_{H} z_{HR}^{2} + 2 d_{H} g_{R} \rho_{H} \rho_{R} z_{HR}^{2} z_{RH} + d_{R}^{2} g_{H}^{2} z_{RH}^{2} + 2 d_{R} g_{H}^{2} g_{R} z_{RH} + 2 d_{R} g_{H}^{2} \rho_{R} z_{RH}^{2} - 2 d_{R} g_{H}^{2} \tau_{R} z_{RH}^{2} + 2 d_{R} g_{H} g_{R} \rho_{H} z_{HR} z_{RH} + g_{H}^{2} g_{R}^{2} + 2 g_{H}^{2} g_{R} \rho_{R} z_{RH} - 2 g_{H}^{2} g_{R} \tau_{R} z_{RH} + g_{H}^{2} \rho_{R}^{2} z_{RH}^{2} - 2 g_{H}^{2} \rho_{R} \tau_{R} z_{RH}^{2} + g_{H}^{2} \tau_{R}^{2} z_{RH}^{2} + 2 g_{H} g_{R}^{2} \rho_{H} z_{HR} + 2 g_{H} g_{R} \rho_{H} \rho_{R} z_{HR} z_{RH} + 2 g_{H} g_{R} \rho_{H} \tau_{R} z_{HR} z_{RH} + g_{R}^{2} \rho_{H}^{2} z_{HR}^{2}}}{2 \left(d_{H} d_{R} z_{HR} + d_{H} \rho_{R} z_{HR} + d_{R} g_{H} + d_{R} \rho_{H} z_{HR} + g_{H} \rho_{R} - g_{H} \tau_{R} + \rho_{H} \rho_{R} z_{HR}\right)} \\
C^{*} &= \frac{g_{C} \left(H^{*} + z_{HC}\right) \left(R^{*} + z_{RC}\right)}{H^{*} R^{*} d_{C} + H^{*} R^{*} \rho_{C} - H^{*} R^{*} \tau_{C} + H^{*} d_{C} z_{RC} - H^{*} \tau_{C} z_{RC} + R^{*} d_{C} z_{HC} + R^{*} \rho_{C} z_{HC} + d_{C} z_{HC} z_{RC}} \\
B^{*} &= \frac{g_{B} \left(H^{*} + z_{HB}\right) \left(R^{*} + z_{RB}\right)}{H^{*} R^{*} d_{B} + H^{*} R^{*} \rho_{B} - H^{*} R^{*} \tau_{B} + H^{*} d_{B} z_{RB} - H^{*} \tau_{B} z_{RB} + R^{*} d_{B} z_{HB} + R^{*} \rho_{B} z_{HB} + d_{B} z_{HB} z_{RB}}
\end{aligned}
$$

### Next-generation matrix

$$
\begin{aligned}
F &= \begin{bmatrix}0 & \frac{g_{E} \tau_{E}}{d_{E} z_{VE}}\\0 & 0\end{bmatrix} \\
V &= \begin{bmatrix}\frac{C^{*} d_{I} + C^{*} \tau_{I} + d_{I} z_{CI}}{C^{*} + z_{CI}} & 0\\- \nu & \frac{B^{*} d_{V} + B^{*} \tau_{V} + d_{V} z_{BV}}{B^{*} + z_{BV}}\end{bmatrix} \\
K = F V^{-1} &= \begin{bmatrix}\frac{g_{E} \nu \tau_{E} \left(B^{*} + z_{BV}\right) \left(C^{*} + z_{CI}\right)}{d_{E} z_{VE} \left(B^{*} d_{V} + B^{*} \tau_{V} + d_{V} z_{BV}\right) \left(C^{*} d_{I} + C^{*} \tau_{I} + d_{I} z_{CI}\right)} & \frac{g_{E} \tau_{E} \left(B^{*} + z_{BV}\right)}{d_{E} z_{VE} \left(B^{*} d_{V} + B^{*} \tau_{V} + d_{V} z_{BV}\right)}\\0 & 0\end{bmatrix}
\end{aligned}
$$

Only $I$ receive new infections, so $\mathcal{R}_0$ is the spectral radius of the next-generation matrix with small domain:

$$
\begin{aligned}
K_{\text{small}} &= \begin{bmatrix}\frac{g_{E} \nu \tau_{E} \left(B^{*} + z_{BV}\right) \left(C^{*} + z_{CI}\right)}{d_{E} z_{VE} \left(B^{*} d_{V} + B^{*} \tau_{V} + d_{V} z_{BV}\right) \left(C^{*} d_{I} + C^{*} \tau_{I} + d_{I} z_{CI}\right)}\end{bmatrix}
\end{aligned}
$$

### Basic reproduction number

$$
\begin{aligned}
\mathcal{R}_0 = \rho\left(F V^{-1}\right) &= \frac{g_{E} \nu \tau_{E} \left(B^{*} + z_{BV}\right) \left(C^{*} + z_{CI}\right)}{d_{E} z_{VE} \left(B^{*} d_{V} + B^{*} \tau_{V} + d_{V} z_{BV}\right) \left(C^{*} d_{I} + C^{*} \tau_{I} + d_{I} z_{CI}\right)}
\end{aligned}
$$

### Numerical values

Illustrative values (not fitted to data). `dfe_numeric` evaluates the definitions in order and `R0_numeric` computes the spectral radius of $K$ with those values.

In [10]:
values = dict(
    g_E=1.0, d_E=0.1, tau_E=2.0, z_VE=1.0, d_I=0.5, tau_I=0.5, z_CI=1.0,
    nu=10.0, d_V=1.0, tau_V=0.5, z_BV=1.0,
    g_H=1.0, d_H=1.0, tau_H=0.5, rho_H=0.3, z_IH=1.0, z_RH=1.0,
    g_C=1.0, d_C=1.0, tau_C=0.5, rho_C=0.3, z_HC=1.0, z_RC=1.0,
    g_B=1.0, d_B=1.0, tau_B=0.5, rho_B=0.3, z_HB=1.0, z_RB=1.0,
    g_R=1.0, d_R=1.0, tau_R=0.5, rho_R=0.3, z_HR=1.0,
)

print("Numerical DFE:")
for x, value in model.dfe_numeric(values).items():
    print(f"  {x} = {value:.6f}")
print(f"R0 = {model.R0_numeric(values):.6f}")

stab = model.dfe_stability(values)
print("Condition (A5): eigenvalues =",
      ", ".join(f"{z.real:.4f}" for z in stab["eigenvalues"]),
      "->", "stable" if stab["stable"] else "unstable")

Numerical DFE:
  E = 10.000000
  I = 0.000000
  V = 0.000000
  H = 0.873253
  C = 1.096420
  B = 1.096420
  R = 0.937281
R0 = 208.197016


Condition (A5): eigenvalues = -0.9121, -0.9121, -1.1060, -1.1060, -0.1000 -> stable


### Threshold

If (A5) holds, the DFE of the full system is locally asymptotically stable when $\mathcal{R}_0 < 1$ and unstable when $\mathcal{R}_0 > 1$. We check this with the largest real part of the eigenvalues of the full ($7\times7$) Jacobian at the DFE, varying $\tau_E$.

In [11]:
f = sp.Matrix([model.equations[v] for v in model.variables])
J = f.jacobian(list(model.variables))

for tau_E in [0.005, 0.01, 2.0]:
    v = dict(values, tau_E=tau_E)
    Jn = J.subs(model.dfe_numeric(v)).subs({s: v[s.name] for s in J.free_symbols - set(model.variables)})
    abscissa = np.max(np.linalg.eigvals(np.array(Jn.evalf(), dtype=float)).real)
    print(f"tau_E = {tau_E:<6} R0 = {model.R0_numeric(v):9.4f}   max Re(λ) = {abscissa:+.4f}")

tau_E = 0.005  R0 =    0.5205   max Re(λ) = -0.1000
tau_E = 0.01   R0 =    1.0410   max Re(λ) = +0.0193


tau_E = 2.0    R0 =  208.1970   max Re(λ) = +13.1328


### Global sensitivity

Global sensitivity does not use the explicit expression of $\mathcal{R}_0$: it evaluates the chain $H^*, R^* \to C^*, B^* \to \mathcal{R}_0$, vectorized with NumPy, and obtains the derivatives (to check the monotonicity required by PRCC) by the chain rule.

In [12]:
t0 = time.time()
res = model.prcc(baseline=values, spread=0.2, n=1000, seed=1)   # all uniform within ±20 %
print(f"LHS-PRCC (n = 1000) in {time.time() - t0:.1f} s; the 8 most influential parameters:")
table = res.as_dict()
for name in res.ranking()[:8]:
    print(f"  {name:<7} PRCC = {table[name]['PRCC']:+.3f}")

LHS-PRCC (n = 1000) in 3.0 s; the 8 most influential parameters:
  nu      PRCC = +0.885
  z_VE    PRCC = -0.882
  g_E     PRCC = +0.881
  d_E     PRCC = -0.880
  tau_E   PRCC = +0.879
  d_V     PRCC = -0.828
  d_I     PRCC = -0.772
  tau_I   PRCC = -0.540


## 3. A DFE without closed form

If a block of the DFE has no closed form (here, a root of $\Lambda - \mu S - kS^5 = 0$), it is kept implicit: `dfe_definitions` marks it with `None`, $\mathcal{R}_0$ is written with $S^*$, and `dfe_numeric` / `R0_numeric` solve it numerically (integration towards steady state and Newton refinement).

In [13]:
model_q = R0Model('''
    dS/dt = Lambda - mu*S - k*S^5 - beta*S*I
    dI/dt = beta*S*I - (gamma + mu)*I
''', infected=["I"])

print("Implicit DFE:", model_q.dfe_is_implicit, "->", model_q.dfe_definitions)
display(Markdown(f"$$\\mathcal{{R}}_0 = {model_q.latex()}$$"))

v = dict(Lambda=2.0, mu=0.5, k=0.1, beta=1.5, gamma=0.3)
S_star = model_q.dfe_numeric(v)[sp.Symbol("S")]
print(f"S* = {S_star:.10f}   (residual: {2 - 0.5*S_star - 0.1*S_star**5:.1e})")
print(f"R0 = {model_q.R0_numeric(v):.10f}")

Implicit DFE: True -> {S_star: None}


$$\mathcal{R}_0 = \frac{S^{*} \beta}{\gamma + \mu}$$

S* = 1.6384507858   (residual: 2.2e-16)
R0 = 3.0720952233
